# Stage 2 — deep models on the best Stage-1 methods

Trains networks on the top methods from Stage 1 and compares **architectures**.

| architecture | note |
|---|---|
| `tremor_bilstm` | the repo's tuned BiLSTM |
| `bilstm`, `gru` | plain recurrent baselines |
| `restcn` | residual temporal conv net |
| `resnet18` | pretrained 2-D CNN on the spectrogram image |
| `resbilstm` | conv front-end + BiLSTM |

**Guards built in**, because each has burned this project before:
* patient-grouped CV, every patient tested once, no recording-level leakage;
* per-recording probabilities aggregated to patient level, so numbers are
  comparable to Stage 1;
* **multi-seed** — a single run once read 0.903 where the 4-seed mean was 0.866;
* **balanced accuracy** on PD-vs-ET (majority baseline 0.833);
* paired bootstrap CI between architectures.

CPU-only this is slow. Start with 2 seeds, 3 folds, 20 epochs to get a signal,
then scale up.

## 1. Setup

In [ ]:
import sys, os, json
if os.path.basename(os.getcwd()) == "tfbench": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from tremor.quaternion_data import load_quaternion_recordings
from tfbench import deep as D

DATA_ROOT, ACTION = "Data", "OUT"
recs = load_quaternion_recordings(DATA_ROOT, action=ACTION, mode="angular_velocity")
try:
    TOP_METHODS = json.load(open("artifacts/tfbench_top_methods.json"))
except FileNotFoundError:
    TOP_METHODS = ["stft256", "cwt", "multitaper", "wavelet_packet"]
TOP_METHODS = [m for m in TOP_METHODS if m in D.METHOD_TO_TFD]
print("methods:", TOP_METHODS)
print("architectures available:", sorted(__import__('tremor.models', fromlist=['MODELS']).MODELS))

## 2. Quick smoke run (small settings — confirm it trains end to end)

In [ ]:
smoke = D.compare(recs, methods=TOP_METHODS[:1], archs=["tremor_bilstm"],
                  axis="PD_vs_ET", seeds=(0,), n_splits=3,
                  epochs=5, patience=3)

## 3. The grid: methods x architectures

Set `EPOCHS`/`SEEDS`/`FOLDS` to what your machine can afford. Every cell of the
grid is `len(seeds) * n_splits` network trainings, so cost grows fast.

In [ ]:
ARCHS  = ["tremor_bilstm", "restcn", "resnet18"]
SEEDS  = (0, 1)
FOLDS  = 5
EPOCHS = 30

res_pe = D.compare(recs, methods=TOP_METHODS, archs=ARCHS, axis="PD_vs_ET",
                   seeds=SEEDS, n_splits=FOLDS, epochs=EPOCHS, patience=8)

## 4. Same grid on the easier axis

In [ ]:
res_nt = D.compare(recs, methods=TOP_METHODS, archs=ARCHS, axis="N_vs_Tremor",
                   seeds=SEEDS, n_splits=FOLDS, epochs=EPOCHS, patience=8)

## 5. Heatmap: method x architecture

Print the seed **spread**, not just the mean — if sd is comparable to the
differences between cells, the grid is not resolving anything.

In [ ]:
def heat(res, title):
    if not res: print("no results for", title); return
    meths = sorted({k[0] for k in res}); archs = sorted({k[1] for k in res})
    M = np.full((len(meths), len(archs)), np.nan)
    S = np.full_like(M, np.nan)
    for (m, a), v in res.items():
        M[meths.index(m), archs.index(a)] = np.mean(v["bal_acc"])
        S[meths.index(m), archs.index(a)] = np.std(v["bal_acc"])
    fig, ax = plt.subplots(figsize=(1.9*len(archs)+3, 1.0*len(meths)+2.2))
    im = ax.imshow(M, cmap="viridis"); fig.colorbar(im, label="balanced accuracy")
    ax.set_xticks(range(len(archs))); ax.set_xticklabels(archs, rotation=30, ha="right")
    ax.set_yticks(range(len(meths))); ax.set_yticklabels(meths)
    for i in range(len(meths)):
        for j in range(len(archs)):
            if not np.isnan(M[i, j]):
                ax.text(j, i, f"{M[i,j]:.3f}\n±{S[i,j]:.3f}", ha="center",
                        va="center", color="w", fontsize=8)
    ax.set_title(title); plt.tight_layout(); plt.show()

heat(res_pe, "PD vs ET (balanced accuracy, majority baseline 0.833)")
heat(res_nt, "N vs Tremor (balanced accuracy)")

## 6. Paired comparison between the two best cells

In [ ]:
from tfbench.benchmark import _bal
order = sorted(res_pe.items(), key=lambda kv: -np.mean(kv[1]["bal_acc"]))
if len(order) >= 2:
    (ka, va), (kb, vb) = order[0], order[1]
    Pa, ya, pa = va["runs"][0]; Pb, yb, pb = vb["runs"][0]
    assert (pa == pb).all() and (ya == yb).all(), "patient sets differ"
    ca = (Pa.argmax(1) == ya).astype(float); cb = (Pb.argmax(1) == yb).astype(float)
    rng = np.random.default_rng(0); idx = np.arange(len(ya))
    d = np.array([ca[s].mean() - cb[s].mean()
                  for s in (rng.choice(idx, len(idx), True) for _ in range(3000))])
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"{ka} vs {kb}")
    print(f"  diff {d.mean():+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  "
          f"{'DISTINGUISHABLE' if lo > 0 else 'not distinguishable'}")

## 7. Compare against the Stage-1 classical result

The honest question is not "which architecture wins" but **"does any deep model
beat the simple descriptor model from Stage 1?"** If not, the extra capacity is
not buying anything on this cohort and the classical model is the one to
report.

In [ ]:
from tfbench import benchmark as B
tab = B.patient_table(recs, "stft256")
Xa, ya, ga = B._axis_data(*tab, "PD_vs_ET")
cls_pred = B._loso(Xa, ya, ga)
print(f"Stage-1 classical (stft256 descriptors): bal-acc {B._bal(ya, cls_pred):.3f}")
if res_pe:
    best = max(res_pe.items(), key=lambda kv: np.mean(kv[1]['bal_acc']))
    print(f"Stage-2 best deep {best[0]}: bal-acc {np.mean(best[1]['bal_acc']):.3f} "
          f"± {np.std(best[1]['bal_acc']):.3f}")